In [62]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

In [63]:
RAW_DATA_DIR = Path("../data/raw")

In [64]:
with open(RAW_DATA_DIR / "recovery.json") as f: 
    recovery_raw = json.load(f)

with open(RAW_DATA_DIR / "sleep.json") as f:
    sleep_raw = json.load(f)

with open(RAW_DATA_DIR / "workouts.json") as f:
    workouts_raw = json.load(f)

with open(RAW_DATA_DIR / "cycles.json") as f:
    cycles_raw = json.load(f)

print(f"Recovery records: {len(recovery_raw)}")
print(f"Sleep records: {len(sleep_raw)}")
print(f"Workout records: {len(workouts_raw)}")
print(f"Cycle records: {len(cycles_raw)}")

Recovery records: 245
Sleep records: 259
Workout records: 381
Cycle records: 246


In [65]:
recovery_raw[0]

{'cycle_id': 1550475332,
 'sleep_id': '977c3bf6-7d11-489e-9d75-b7f1973cc48a',
 'user_id': 31526255,
 'created_at': '2026-06-06T13:18:35.860Z',
 'updated_at': '2026-06-06T13:18:35.860Z',
 'score_state': 'SCORED',
 'score': {'user_calibrating': False,
  'recovery_score': 55.0,
  'resting_heart_rate': 45.0,
  'hrv_rmssd_milli': 118.53185,
  'spo2_percentage': 96.111115,
  'skin_temp_celsius': 33.024}}

In [66]:
# check out one record from each
print("SLEEP:")
print(list(sleep_raw[0].keys()))
print("\nSLEEP SCORE KEYS:")
print(list(sleep_raw[0]['score'].keys()))

print("\nWORKOUT:")
print(list(workouts_raw[0].keys()))

print("\nCYCLE:")
print(list(cycles_raw[0].keys()))

SLEEP:
['id', 'cycle_id', 'v1_id', 'user_id', 'created_at', 'updated_at', 'start', 'end', 'timezone_offset', 'nap', 'score_state', 'score']

SLEEP SCORE KEYS:
['stage_summary', 'sleep_needed', 'respiratory_rate', 'sleep_performance_percentage', 'sleep_consistency_percentage', 'sleep_efficiency_percentage']

WORKOUT:
['id', 'v1_id', 'user_id', 'created_at', 'updated_at', 'start', 'end', 'timezone_offset', 'sport_name', 'score_state', 'score', 'sport_id']

CYCLE:
['id', 'user_id', 'created_at', 'updated_at', 'start', 'end', 'timezone_offset', 'score_state', 'score']


In [67]:
sleep_raw[0]['score']['stage_summary']

{'total_in_bed_time_milli': 30525730,
 'total_awake_time_milli': 1758040,
 'total_no_data_time_milli': 0,
 'total_light_sleep_time_milli': 14685350,
 'total_slow_wave_sleep_time_milli': 6006130,
 'total_rem_sleep_time_milli': 8076210,
 'sleep_cycle_count': 5,
 'disturbance_count': 10}

In [68]:
recovery_df = pd.json_normalize(recovery_raw)

print(recovery_df.shape)
print(recovery_df.columns.tolist())

(245, 12)
['cycle_id', 'sleep_id', 'user_id', 'created_at', 'updated_at', 'score_state', 'score.user_calibrating', 'score.recovery_score', 'score.resting_heart_rate', 'score.hrv_rmssd_milli', 'score.spo2_percentage', 'score.skin_temp_celsius']


In [69]:
# normalize all datasets
sleep_df = pd.json_normalize(sleep_raw)
workouts_df = pd.json_normalize(workouts_raw)
cycles_df = pd.json_normalize(cycles_raw)

# filter to only SCORED records
recovery_df = recovery_df[recovery_df['score_state'] == 'SCORED']
sleep_df = sleep_df[sleep_df['score_state'] == 'SCORED']
workouts_df = workouts_df[workouts_df['score_state'] == 'SCORED']
cycles_df = cycles_df[cycles_df['score_state'] == 'SCORED']

# filter out naps from sleep - keep only main sleep sessions
sleep_df = sleep_df[sleep_df['nap'] == False]

print(f"Recovery after filtering: {len(recovery_df)}")
print(f"Sleep after filtering: {len(sleep_df)}")
print(f"Workouts after filtering: {len(workouts_df)}")
print(f"Cycles after filtering: {len(cycles_df)}")

Recovery after filtering: 245
Sleep after filtering: 245
Workouts after filtering: 381
Cycles after filtering: 246


In [70]:
recovery_df['created_at'] = pd.to_datetime(recovery_df['created_at'])
cycles_df['start'] = pd.to_datetime(cycles_df['start'])
sleep_df['start'] = pd.to_datetime(sleep_df['start'])
sleep_df['end'] = pd.to_datetime(sleep_df['end'])
workouts_df['start'] = pd.to_datetime(workouts_df['start'])
workouts_df['end'] = pd.to_datetime(workouts_df['end'])

# add a clean date column to recovery = primary index
recovery_df['date'] = recovery_df['created_at'].dt.date
recovery_df = recovery_df.sort_values('date').reset_index(drop=True)

print(f"Date range: {recovery_df['date'].min()} to {recovery_df['date'].max()}")
print(f"Total days: {(recovery_df['date'].max() - recovery_df['date'].min()).days}")

Date range: 2025-10-05 to 2026-06-06
Total days: 244


In [71]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08)

# HRV
fig.add_trace(go.Scatter(
    x=recovery_df['date'], 
    y=recovery_df['score.hrv_rmssd_milli'],
    mode='lines', name='HRV (raw)',
    line=dict(color='#6366f1', width=1),
    opacity=0.4
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=recovery_df['date'],
    y=recovery_df['score.hrv_rmssd_milli'].rolling(7).mean(),
    mode='lines', name='HRV (7-day avg)',
    line=dict(color='#6366f1', width=2)
), row=1, col=1)

# Recovery
fig.add_trace(go.Scatter(
    x=recovery_df['date'],
    y=recovery_df['score.recovery_score'],
    mode='lines', name='Recovery (raw)',
    line=dict(color='#10b981', width=1),
    opacity=0.4
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=recovery_df['date'],
    y=recovery_df['score.recovery_score'].rolling(7).mean(),
    mode='lines', name='Recovery (7-day avg)',
    line=dict(color='#10b981', width=2)
), row=2, col=1)

fig.update_layout(height=600, title='HRV and Recovery over time')
fig.update_yaxes(title_text='HRV (ms)', row=1, col=1)
fig.update_yaxes(title_text='Recovery score', row=2, col=1)

fig.show()

In [72]:
import ruptures as rpt
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# load processed daily data
daily_df = pd.read_json("../data/processed/daily.json")
daily_df["date"] = pd.to_datetime(daily_df["date"])

# raw HRV signal
raw_signal = daily_df["hrv"].dropna().values
raw_dates = daily_df["date"][daily_df["hrv"].notna()].values

# z-score normalize — makes penalty scale-invariant across users
signal_mean = raw_signal.mean()
signal_std = raw_signal.std()
z_signal = (raw_signal - signal_mean) / signal_std

print(f"Signal length: {len(raw_signal)}")
print(f"Original — mean: {signal_mean:.1f}ms, std: {signal_std:.1f}ms")
print(f"Z-scored — mean: {z_signal.mean():.3f}, std: {z_signal.std():.3f}")

Signal length: 244
Original — mean: 126.6ms, std: 16.9ms
Z-scored — mean: -0.000, std: 1.000


In [73]:
# sweep penalties to understand sensitivity
# looking for segments that are physiologically and biographically meaningful
model = rpt.Pelt(model="rbf").fit(z_signal)

print("Penalty sweep:")
for pen in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 5.0]:
    bkps = model.predict(pen=pen)
    dates = [str(raw_dates[i-1])[:10] for i in bkps[:-1]]
    print(f"  pen={pen}: {len(bkps)} segments → {dates}")

Penalty sweep:
  pen=0.5: 27 segments → ['2025-10-09', '2025-10-14', '2025-10-19', '2025-11-08', '2025-11-13', '2025-11-18', '2025-11-28', '2025-12-18', '2025-12-23', '2026-01-02', '2026-01-12', '2026-01-17', '2026-01-27', '2026-02-01', '2026-02-11', '2026-02-26', '2026-03-08', '2026-03-29', '2026-04-03', '2026-04-13', '2026-04-18', '2026-04-23', '2026-04-28', '2026-05-08', '2026-05-23', '2026-05-28']
  pen=1.0: 14 segments → ['2025-10-14', '2025-11-08', '2025-12-18', '2025-12-23', '2026-01-02', '2026-01-12', '2026-02-01', '2026-03-08', '2026-03-29', '2026-04-08', '2026-04-23', '2026-05-08', '2026-05-28']
  pen=1.5: 12 segments → ['2025-10-14', '2025-11-08', '2025-12-18', '2025-12-23', '2026-01-02', '2026-01-12', '2026-02-01', '2026-03-08', '2026-04-23', '2026-05-08', '2026-05-28']
  pen=2.0: 11 segments → ['2025-10-14', '2025-11-08', '2025-12-18', '2025-12-23', '2026-01-02', '2026-01-12', '2026-02-01', '2026-03-08', '2026-04-23', '2026-05-08']
  pen=2.5: 9 segments → ['2025-11-08', '2

In [74]:
from plotly.subplots import make_subplots

penalties = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
penalty_labels = [f"pen={p}" for p in penalties]
hrv_smooth = pd.Series(raw_signal).rolling(7, min_periods=3).mean()

fig = make_subplots(
    rows=len(penalties), cols=1,
    shared_xaxes=True,
    subplot_titles=penalty_labels,
    vertical_spacing=0.04
)

for idx, pen in enumerate(penalties, start=1):
    bkps = model.predict(pen=pen)

    fig.add_trace(go.Scatter(
        x=raw_dates, y=raw_signal,
        mode='lines', name='HRV (raw)',
        line=dict(color='#6366f1', width=1),
        opacity=0.3, showlegend=(idx == 1)
    ), row=idx, col=1)

    fig.add_trace(go.Scatter(
        x=raw_dates, y=hrv_smooth,
        mode='lines', name='HRV (7d avg)',
        line=dict(color='#6366f1', width=2),
        showlegend=(idx == 1)
    ), row=idx, col=1)

    for bp in bkps[:-1]:
        fig.add_vline(
            x=str(raw_dates[bp])[:10],
            line_dash="dash", line_color="red",
            line_width=1.5, opacity=0.8,
            row=idx, col=1
        )

    fig.add_annotation(
        x=raw_dates[5], y=raw_signal.max(),
        text=f"{len(bkps)} segments",
        showarrow=False,
        font=dict(size=11, color="red"),
        row=idx, col=1
    )

fig.update_layout(
    height=220 * len(penalties),
    title='PELT change-points across penalty values (z-scored HRV)',
    showlegend=True
)
fig.update_yaxes(title_text='HRV (ms)')
fig.show()

## Penalty selection: pen=3.0

BIC penalty (log n ≈ 5.5) was too conservative - produced 0 breakpoints.
pen=2.5 produced 9 segments, over-segmenting the post-graduation period.
pen=3.0 selected as the most physiologically and biographically meaningful:

- Oct–Dec: master's program - structured routine, consistent physiology
- Dec–Jan: holiday crash - travel, disrupted sleep, no structure
- Jan–now: post-graduation - rebuilding, new rhythm

PELT recovered these life phases from HRV alone with no biographical input.
Robust breakpoints (Dec 23, Jan 12) appear across all penalty values from 1.0–3.0,
confirming the January crash is the strongest regime shift in the dataset.

In [75]:
# final model — pen=3.0
bkps = model.predict(pen=3.0)

chapters = []
prev = 0
for i, bp in enumerate(bkps):
    start_date = pd.Timestamp(raw_dates[prev])
    end_date = pd.Timestamp(raw_dates[min(bp-1, len(raw_dates)-1)])
    n_days = bp - prev
    chapters.append({
        "chapter": i + 1,
        "start_idx": prev,
        "end_idx": bp,
        "start_date": start_date,
        "end_date": end_date,
        "n_days": n_days,
    })
    prev = bp

chapter_names = {
    1: "The Student",
    2: "The Crash",
    3: "Post-Grad",
}

print(f"Final chapters (pen=3.0):")
for c in chapters:
    print(f"  {chapter_names[c['chapter']]}: {str(c['start_date'])[:10]} → {str(c['end_date'])[:10]} ({c['n_days']} days)")

Final chapters (pen=3.0):
  The Student: 2025-10-05 → 2025-12-23 (80 days)
  The Crash: 2025-12-24 → 2026-01-12 (20 days)
  Post-Grad: 2026-01-13 → 2026-06-06 (144 days)


In [76]:
# assign chapter to each day
daily_df["chapter"] = None
for c in chapters:
    mask = (daily_df["date"] >= c["start_date"]) & (daily_df["date"] <= c["end_date"])
    daily_df.loc[mask, "chapter"] = c["chapter"]

# compute fingerprint per chapter
fingerprints = daily_df.groupby("chapter").agg(
    mean_hrv=("hrv", "mean"),
    mean_recovery=("recovery_score", "mean"),
    mean_sws=("sws_pct", "mean"),
    mean_rem=("rem_pct", "mean"),
    mean_sleep_efficiency=("sleep_efficiency", "mean"),
    mean_strain=("strain", "mean"),
    mean_load_ratio=("load_ratio", "mean"),
    mean_stability=("stability", "mean"),
    hrv_trend=("hrv", lambda x: "rising" if x.iloc[-1] > x.iloc[0] else "declining"),
    n_days=("hrv", "count"),
).reset_index()

fingerprints["name"] = fingerprints["chapter"].map(chapter_names)

print(fingerprints[[
    "chapter", "name", "mean_hrv", "mean_recovery",
    "mean_strain", "mean_stability", "hrv_trend"
]].to_string(index=False))

 chapter        name   mean_hrv  mean_recovery  mean_strain  mean_stability hrv_trend
       1 The Student 134.152693      71.787500    10.315528       90.410851 declining
       2   The Crash 105.979493      51.100000     9.645774       87.570549    rising
       3   Post-Grad 125.331151      69.756944    11.275616       88.227987    rising


In [77]:
chapter_colors = {
    1: "rgba(99, 102, 241, 0.12)",
    2: "rgba(239, 68, 68, 0.15)",
    3: "rgba(16, 185, 129, 0.12)",
}

chapter_label_colors = {
    1: "#6366f1",
    2: "#ef4444",
    3: "#10b981",
}

fig = go.Figure()

# shaded regions
for c in chapters:
    fig.add_vrect(
        x0=str(c["start_date"])[:10],
        x1=str(c["end_date"])[:10],
        fillcolor=chapter_colors[c["chapter"]],
        layer="below",
        line_width=0,
    )
    mid_date = c["start_date"] + (c["end_date"] - c["start_date"]) / 2
    fig.add_annotation(
        x=mid_date,
        y=raw_signal.max() + 5,
        text=chapter_names[c["chapter"]],
        showarrow=False,
        font=dict(size=13, color=chapter_label_colors[c["chapter"]]),
    )

# raw HRV
fig.add_trace(go.Scatter(
    x=raw_dates, y=raw_signal,
    mode='lines', name='HRV (raw)',
    line=dict(color='#6366f1', width=1),
    opacity=0.3,
))

# smoothed HRV
fig.add_trace(go.Scatter(
    x=raw_dates, y=hrv_smooth,
    mode='lines', name='HRV (7d avg)',
    line=dict(color='#6366f1', width=2.5),
))

# breakpoint lines
for bp in bkps[:-1]:
    fig.add_vline(
        x=str(raw_dates[bp])[:10],
        line_dash="dash",
        line_color="#94a3b8",
        line_width=1.5,
    )

fig.update_layout(
    title='Your health arc — three chapters',
    height=500,
    yaxis_title='HRV (ms)',
    hovermode='x unified',
)

fig.show()

In [78]:
# anomaly detection — legendary days and boss fights
# composite signal: equal weight of recovery score and autonomic recovery

recovery_z = (daily_df["recovery_score"] - daily_df["recovery_score"].mean()) / daily_df["recovery_score"].std()
autonomic_z = (daily_df["autonomic_recovery"] - daily_df["autonomic_recovery"].mean()) / daily_df["autonomic_recovery"].std()
daily_df["composite"] = 0.5 * recovery_z + 0.5 * autonomic_z # average of two z-scores

# z-score against 30-day rolling baseline
# rolling rather than global - anomalies are relative to recent state, not all-time
rolling_mean = daily_df["composite"].rolling(window=30, min_periods=7).mean()
rolling_std = daily_df["composite"].rolling(window=30, min_periods=7).std()

daily_df["composite_z"] = (daily_df["composite"] - rolling_mean) / rolling_std

# flag anomalies
LEGENDARY_THRESHOLD = 1.5
BOSS_THRESHOLD = -1.5

legendary = daily_df[daily_df["composite_z"] > LEGENDARY_THRESHOLD].copy()
boss_fights = daily_df[daily_df["composite_z"] < BOSS_THRESHOLD].copy()

print(f"Legendary days: {len(legendary)}")
print(f"Boss fights: {len(boss_fights)}")
print()
print("Top 5 legendary days:")
print(legendary.nlargest(5, "composite_z")[["date", "recovery_score", "hrv", "sws_pct", "rem_pct", "composite_z"]].to_string(index=False))
print()
print("Top 5 boss fights:")
print(boss_fights.nsmallest(5, "composite_z")[["date", "recovery_score", "hrv", "sws_pct", "rem_pct", "composite_z"]].to_string(index=False))

Legendary days: 17
Boss fights: 15

Top 5 legendary days:
      date  recovery_score       hrv   sws_pct   rem_pct  composite_z
2026-01-17              86 144.26395 27.416156 24.792021     2.158717
2026-01-14              91 135.80167 27.119297 27.011074     1.984717
2025-11-11              95 151.85520 25.660683 26.084888     1.947669
2026-01-19              90 146.88210 31.784208 28.100059     1.929397
2025-12-16              95 176.85654 30.397145 29.099903     1.871316

Top 5 boss fights:
      date  recovery_score        hrv   sws_pct   rem_pct  composite_z
2026-05-01              12  71.053055 26.430454 30.719986    -3.314722
2026-04-05              30  80.680640 26.305269 19.936022    -2.525424
2025-12-30              32  95.310220 30.510743 27.473542    -2.196432
2026-01-07              24  71.813930 21.161234 23.532551    -2.090349
2025-12-24              40 111.104350 23.910293 40.958168    -1.984189


In [79]:
for threshold in [1.0, 1.5, 2.0, 2.5]:
    leg = len(daily_df[daily_df["composite_z"] > threshold])
    boss = len(daily_df[daily_df["composite_z"] < -threshold])
    print(f"threshold={threshold}: {leg} legendary, {boss} boss fights")

threshold=1.0: 50 legendary, 42 boss fights
threshold=1.5: 17 legendary, 15 boss fights
threshold=2.0: 1 legendary, 4 boss fights
threshold=2.5: 0 legendary, 2 boss fights


In [80]:
# top 5 only for cleaner visualization
top_legendary = legendary.nlargest(5, "composite_z")
top_boss = boss_fights.nsmallest(5, "composite_z")

hrv_smooth = daily_df["hrv"].rolling(7, min_periods=3).mean()

fig = go.Figure()

# chapter shading — very subtle
for c in chapters:
    fig.add_vrect(
        x0=str(c["start_date"])[:10],
        x1=str(c["end_date"])[:10],
        fillcolor=chapter_colors[c["chapter"]],
        layer="below",
        line_width=0,
        opacity=0.4,
    )

# raw HRV
fig.add_trace(go.Scatter(
    x=daily_df["date"],
    y=daily_df["hrv"],
    mode='lines',
    name='HRV (raw)',
    line=dict(color='#6366f1', width=0.8),
    opacity=0.2,
))

# smoothed HRV
fig.add_trace(go.Scatter(
    x=daily_df["date"],
    y=hrv_smooth,
    mode='lines',
    name='HRV (7d avg)',
    line=dict(color='#6366f1', width=2.5),
))

# legendary days
fig.add_trace(go.Scatter(
    x=top_legendary["date"],
    y=top_legendary["hrv"],
    mode='markers+text',
    name='Legendary day',
    textposition='top center',
    marker=dict(color='gold', size=12, symbol='star',
                line=dict(color='darkorange', width=1)),
))

# boss fights
fig.add_trace(go.Scatter(
    x=top_boss["date"],
    y=top_boss["hrv"],
    mode='markers+text',
    name='Boss fight',
    textposition='bottom center',
    marker=dict(color='#ef4444', size=12, symbol='x',
                line=dict(color='darkred', width=2)),
))

fig.update_layout(
    title='Your health arc — legendary days and boss fights',
    height=500,
    yaxis_title='HRV (ms)',
    plot_bgcolor='white',
    hovermode='x unified',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()

In [81]:
import json

PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# save chapters
chapters_output = []
for c in chapters:
    # get fingerprint for this chapter
    fp = fingerprints[fingerprints["chapter"] == c["chapter"]].iloc[0]
    chapters_output.append({
        "chapter": c["chapter"],
        "name": chapter_names[c["chapter"]],
        "start_date": str(c["start_date"])[:10],
        "end_date": str(c["end_date"])[:10],
        "n_days": c["n_days"],
        "mean_hrv": round(float(fp["mean_hrv"]), 1),
        "mean_recovery": round(float(fp["mean_recovery"]), 1),
        "mean_sws": round(float(fp["mean_sws"]), 1),
        "mean_rem": round(float(fp["mean_rem"]), 1),
        "mean_sleep_efficiency": round(float(fp["mean_sleep_efficiency"]), 1),
        "mean_strain": round(float(fp["mean_strain"]), 1),
        "mean_stability": round(float(fp["mean_stability"]), 1),
        "hrv_trend": fp["hrv_trend"],
    })

with open(PROCESSED_DATA_DIR / "chapters.json", "w") as f:
    json.dump(chapters_output, f, indent=2)

print("chapters.json saved")
print(json.dumps(chapters_output, indent=2))

chapters.json saved
[
  {
    "chapter": 1,
    "name": "The Student",
    "start_date": "2025-10-05",
    "end_date": "2025-12-23",
    "n_days": 80,
    "mean_hrv": 134.2,
    "mean_recovery": 71.8,
    "mean_sws": 29.7,
    "mean_rem": 27.5,
    "mean_sleep_efficiency": 94.8,
    "mean_strain": 10.3,
    "mean_stability": 90.4,
    "hrv_trend": "declining"
  },
  {
    "chapter": 2,
    "name": "The Crash",
    "start_date": "2025-12-24",
    "end_date": "2026-01-12",
    "n_days": 20,
    "mean_hrv": 106.0,
    "mean_recovery": 51.1,
    "mean_sws": 24.6,
    "mean_rem": 29.2,
    "mean_sleep_efficiency": 94.6,
    "mean_strain": 9.6,
    "mean_stability": 87.6,
    "hrv_trend": "rising"
  },
  {
    "chapter": 3,
    "name": "Post-Grad",
    "start_date": "2026-01-13",
    "end_date": "2026-06-06",
    "n_days": 144,
    "mean_hrv": 125.3,
    "mean_recovery": 69.8,
    "mean_sws": 27.0,
    "mean_rem": 26.0,
    "mean_sleep_efficiency": 94.4,
    "mean_strain": 11.3,
    "mean_st

In [83]:
# save legendary days
legendary_output = legendary.sort_values("composite_z", ascending=False)[[
    "date", "recovery_score", "hrv", "sws_pct", "rem_pct", 
    "sleep_efficiency", "strain", "autonomic_recovery", "composite_z"
]].copy()

legendary_output["date"] = legendary_output["date"].astype(str)
legendary_output = legendary_output.round(2)

# save boss fights
boss_output = boss_fights.sort_values("composite_z", ascending=True)[[
    "date", "recovery_score", "hrv", "sws_pct", "rem_pct",
    "sleep_efficiency", "strain", "autonomic_recovery", "composite_z"
]].copy()

boss_output["date"] = boss_output["date"].astype(str)
boss_output = boss_output.round(2)

# save to disk
with open(PROCESSED_DATA_DIR / "legendary.json", "w") as f:
    json.dump(legendary_output.to_dict(orient="records"), f, indent=2)

with open(PROCESSED_DATA_DIR / "boss_fights.json", "w") as f:
    json.dump(boss_output.to_dict(orient="records"), f, indent=2)

print(f"legendary.json saved - {len(legendary_output)} days")
print(f"boss_fights.json saved - {len(boss_output)} days")

legendary.json saved - 17 days
boss_fights.json saved - 15 days
